# NVIDIA LocateAnything-3B - small exploratory test

Not gated (unlike SAM3/3.1, which need manual HF approval - held off separately). Testing
on a single frame only, 4-bit quantized proactively: NVIDIA's own numbers show ~35GB peak
memory with dense attention vs ~11.7GB with FlashAttention, and Kaggle's T4 (Turing arch)
doesn't properly support the FlashAttention backend that gets it down to 11.7GB - same OOM
risk pattern hit earlier tonight with Qwen3-VL on this same GPU class. Quantizing up front
rather than waiting to hit the same wall again.

This is exploratory ("within budget" - one frame, not a full benchmark) - a VLM grounding
call is a fundamentally different mechanism from the YOLO+SAHI detectors already benchmarked,
not a drop-in comparison.

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes huggingface_hub

In [ ]:
import glob
import cv2
from PIL import Image

found = glob.glob("/kaggle/input/flytbase-hackathon-footage/**/*.MP4", recursive=True)
assert found, "No .MP4 found under the mounted dataset"
cap = cv2.VideoCapture(found[0])
cap.set(cv2.CAP_PROP_POS_FRAMES, 4570)  # same frame used in the benchmark visual comparison
ok, frame = cap.read()
cap.release()
assert ok
image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
print("Frame size:", image.size)
image.save("/kaggle/working/test_frame.jpg")

In [ ]:
from transformers import AutoModel, AutoTokenizer, AutoProcessor, BitsAndBytesConfig

model_path = "nvidia/LocateAnything-3B"
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
model = AutoModel.from_pretrained(
    model_path, quantization_config=quant_config, device_map="cuda:0", trust_remote_code=True,
)
print("Loaded OK")

In [ ]:
prompt = "Locate all the instances that matches the following description: car</c>truck</c>bus</c>van</c>motorcycle</c>pedestrian."

messages = [{"role": "user", "content": [
    {"type": "image", "image": image},
    {"type": "text", "text": prompt},
]}]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)

generated = model.generate(**inputs, max_new_tokens=2048, do_sample=False)
answer = processor.batch_decode(generated[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
print(answer)

In [ ]:
import re
from PIL import ImageDraw

def parse_boxes(text, w, h):
    boxes = []
    for m in re.finditer(r"<box><(\d+)><(\d+)><(\d+)><(\d+)></box>", text):
        x1, y1, x2, y2 = [int(g) for g in m.groups()]
        boxes.append((x1 / 1000 * w, y1 / 1000 * h, x2 / 1000 * w, y2 / 1000 * h))
    return boxes

boxes = parse_boxes(answer, *image.size)
print(f"Parsed {len(boxes)} boxes")

annotated = image.copy()
draw = ImageDraw.Draw(annotated)
for x1, y1, x2, y2 in boxes:
    draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
annotated.save("/kaggle/working/locate_anything_result.jpg")
print("Saved: /kaggle/working/locate_anything_result.jpg")